### Установка зависимостей

In [1]:
%pip install -U numpy matplotlib ipywidgets tqdm PyMuPDF raglite raglite[pandoc] lightrag-hku[api] scikit-learn ollama nest_asyncio pyvis rerankers[flashrank]

  Using cached numpy-2.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Note: you may need to restart the kernel to use updated packages.


### Подготовка функций для оценки.
Взято [отсюда](https://www.geeksforgeeks.org/nlp/evaluation-metrics-for-retrieval-augmented-generation-rag-systems/).

In [3]:
import numpy as np

# MRR
def mean_reciprocal_rank(y_true, y_pred):
    reciprocal_ranks = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        rr = 0
        for rank, doc in enumerate(pred_docs, start=1):
            if doc in true_docs:
                rr = 1 / rank
                break
        reciprocal_ranks.append(rr)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)
def ndcg(y_true, y_pred, k=5):
    ndcg_scores = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        pred_docs_k = pred_docs[:k]
        dcg = sum([1 / np.log2(idx + 2) if doc in true_docs else 0 for idx, doc in enumerate(pred_docs_k)])
        ideal_docs_k = true_docs[:k]
        idcg = sum([1 / np.log2(idx + 2) for idx, _ in enumerate(ideal_docs_k)])
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(ndcg_scores)
def recall_precision_at_k(y_true, y_pred, k=5):
    recall_list = []
    precision_list = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        top_k = pred_docs[:k]
        hits = len([doc for doc in top_k if doc in true_docs])
        recall_list.append(hits / len(true_docs) if true_docs else 0)
        precision_list.append(hits / k)
    return np.mean(recall_list), np.mean(precision_list)

# Example usage:
#y_true = [['doc1', 'doc2'], ['doc3']]
#y_pred = [['doc2', 'doc4'], ['doc5']]
#print("nDCG@5:", ndcg(y_true, y_pred))
#will print: nDCG@5: 0.3065735963827292

### Подготовка данных (фильтрация)
Для повышения качества оценки будут использованы данные, экспортированные в MarkDown формат (формулы в TeX-формате) через PaddleOCR (PPv3).
Данные представляют собой статьи по математике из открытого [источника](https://huggingface.co/datasets/PleIAs/Math-PDF/blob/main/math_pdf_tars/openalex_math_pdf_tar_31.tar). Большая часть статей в данном датасете позволяют использование экспорта текста без OCR.
Будет взято меньше 50 статей + 5 статей с релевантной темой для запросов.

Список доп. статей:
* https://math.berkeley.edu/~giventh/papers/qkf.pdf
* https://www.ams.org/journals/jams/2014-27-04/S0894-0347-2014-00797-9/S0894-0347-2014-00797-9.pdf
* https://www.cambridge.org/core/services/aop-cambridge-core/content/view/935C492E469B3B107B20F50DFE0C0F64/S2050509424001476a.pdf/a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
* https://personal.math.vt.edu/lmihalce/QKlectures(MSJ23).pdf
* и W1605366104.pdf из датасета

In [2]:
import os

DATA_DIR="../../data"
RAW_DATA_DIR=f"{DATA_DIR}/openalex_math_pdf_tar_31"
PROCESSED_DATA_DIR=f"{DATA_DIR}/for_rag_3"
DATA_PATH_OCR=os.path.join(PROCESSED_DATA_DIR, "ocr_data")
MATH_LIMIT=2000
FIND_ALL_DOCS_POSTFIX=" Find all relevant documents."
COEF_K=5
MODEL_LLM="qwen3:8b"
MODEL_EMBED="embeddinggemma:300m"

In [2]:
%pip install psutil ninja packaging
%pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.7.2/flash_attn-2.7.4%2Bcu129torch2.8-cp312-cp312-linux_x86_64.whl

!git clone https://github.com/tencent/WeDLM.git
!cd WeDLM && pip install -e .

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 414.4/414.4 MB 33.7 MB/s  0:00:15:00:0100:01
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'WeDLM' already exists and is not an empty directory.
Obtaining file:///home/alexey/test/mipt_mag_diploma/conference/rnd_2/WeDLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for wedlm (pyproject.toml) ... done
  Created wheel for wedlm: filename=wedlm-0.1.0-0.editable-py3-none-any.whl size=13066 sha256=163c4dc65b2b75866221881ec927ea94753e13a803e11dcc446864da46bb09f2
  Stored in directory: /tmp/pip-ephem-wheel-cache-gxvlnb__/wheels/53/e0/07/4f09ef79eac5090e56a46bc554c16e38183e16fe6d27403bd8
Successfully built wedlm
  Attempting uninstall: wedlm
    Found existing i

In [ ]:
from transformers import AutoTokenizer
from wedlm import LLM, SamplingParams

llm = LLM(model="tencent/WeDLM-8B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("tencent/WeDLM-8B-Instruct", trust_remote_code=True)

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

num_kvcache_blocks (0) is <= 0. Setting it to 1.


In [8]:
def req_math(text, prompt_ask="Is it a scientific text about mathematics (not physics or computer science)?", text_limit=1024):
    prompt = f"{prompt_ask} The text to analyze is below:\n--\n{text[:text_limit]}\n--\nReturn only YES or NO answer without ANY explanation."
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = llm.generate([text], SamplingParams(temperature=0.2, max_tokens=512))
    return outputs[0]["text"]

Первый этап фильтрации с простой моделью. Отбрасываем статьи не по математике.

In [9]:
from pathlib import Path
from tqdm.notebook import tqdm
import fitz
from pymupdf import FileDataError
import os
import shutil

tmp_math_files_n = 0

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
for file in tqdm(list(Path(RAW_DATA_DIR).glob("*.pdf"))):
    print(f"{tmp_math_files_n}/{MATH_LIMIT} ", end='')
    if os.path.exists(os.path.join(PROCESSED_DATA_DIR, file.name)):
        print(f"Already processed")
        tmp_math_files_n += 1
        continue
    text = ""
    try:
        with fitz.open(file) as doc:
            for page in doc:  # iterate the document pages
                text += page.get_text()
    except FileDataError as e:
        print(f"Error reading file {file.name}: {e}")
        continue
    res = req_math(text=text)
    #print(f"File: {file.name}, Result: {res}")
    if res[:3] == "YES":
        # just copy file
        print(f"Copying file: {file.name}")
        shutil.copy(file, os.path.join(PROCESSED_DATA_DIR, file.name))
        tmp_math_files_n += 1
        if tmp_math_files_n >= MATH_LIMIT:
            print(f"Reached math file limit of {MATH_LIMIT}. Stopping.")
            break
    elif res[:2] == "NO":
        print(f"Skipping file: {file.name}")
    else:
        print(f"Unexpected result for file {file.name}: {res}")
del tmp_math_files_n
    

  0%|          | 0/5000 [00:00<?, ?it/s]

0/10000 

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

[Stats] decode_tokens=1026, decode_forwards=68, tokens_per_forward=15.0882
Copying file: W4295565825.pdf
1/10000 

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

...

[Stats] decode_tokens=519, decode_forwards=36, tokens_per_forward=14.4167
Copying file: W4280636005.pdf
2807/10000 

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

[Stats] decode_tokens=514, decode_forwards=34, tokens_per_forward=15.1176
Skipping file: W2604115454.pdf
2807/10000 

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

[Stats] decode_tokens=514, decode_forwards=34, tokens_per_forward=15.1176
Copying file: W2977615596_3.pdf


In [10]:
files_to_add = [
    "QKlectures(MSJ23).pdf",
    "qkf.pdf",
    "S0894-0347-2014-00797-9.pdf",
    "a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf"
]

OLD_DATA_DIR=f"{DATA_DIR}/for_rag"

for file_name in files_to_add:
    src_path = os.path.join(OLD_DATA_DIR, file_name)
    dst_path = os.path.join(PROCESSED_DATA_DIR, file_name)
    if os.path.exists(src_path):
        print(f"Adding file: {file_name}")
        shutil.copy(src_path, dst_path)
    else:
        raise FileNotFoundError(f"File to add not found: {file_name}")

Adding file: QKlectures(MSJ23).pdf
Adding file: qkf.pdf
Adding file: S0894-0347-2014-00797-9.pdf
Adding file: a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf


In [4]:
from pathlib import Path

math_files = [ file.name for file in list(Path(PROCESSED_DATA_DIR).glob("*.pdf"))]
print(f"Total math-related files prepared for RAG: {len(math_files)}")
for file in math_files:
    print(f"* {file}")

Total math-related files prepared for RAG: 2812
* W4295565825.pdf
* W2957174555_1.pdf
* W2796609034.pdf
* W4287119660_1.pdf
* W4313001346.pdf
* W4377086487_5.pdf
* W2465613768.pdf
* W4289128375.pdf
* W4226456969_2.pdf
* W4317037187_2.pdf
* W4396577029.pdf
* W3188079605_3.pdf
* W4300932590_1.pdf
* W2551158135.pdf
* W2802454767_1.pdf
* W2223722977_3.pdf
* W2613469027_6.pdf
* W4206656803.pdf
* W3157468823.pdf
* W4324126587_2.pdf
* W4297662594.pdf
* W1603196302_1.pdf
* W2158760784.pdf
* W2519367019.pdf
* W2108242840.pdf
* W2999061577_2.pdf
* W25036734.pdf
* W4285891493.pdf
* W4389349667.pdf
* W2945171940.pdf
* W4323565661_2.pdf
* W4289543381.pdf
* W2171382235.pdf
* W4287667938.pdf
* W3118338062.pdf
* W4319655444_7.pdf
* W2978550104.pdf
* W3211598426.pdf
* W3152618620_1.pdf
* W2616739057.pdf
* W2963449700_2.pdf
* W4394719147.pdf
* W2803496127.pdf
* W4310022274_2.pdf
* W1971454495_1.pdf
* W4376956170.pdf
* W4386002119.pdf
* W2972953549_2.pdf
* W3128183679.pdf
* W4384347552.pdf
* W2035559671.

### OCR

In [1]:
!pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip3 install paddleocr[all]

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.2-cp312-cp312-linux_x86_64.whl (1890.5 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/httpx/httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/numpy/numpy-2.3.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/protobuf/protobuf-6.33.0-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/pillow/pillow-12.0.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.0 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/opt-einsum/opt_einsum-3.3.0-py3-none-any.whl (65 kB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/networkx/networkx-3.5-py3-none-any.whl (2.0 MB)
  Using cached https://pa

In [6]:
%pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [1]:
# https://arxiv.org/pdf/2507.05595
# See https://github.com/PaddlePaddle/PaddleX/pull/4860
from paddleocr import PPStructureV3

pipeline = PPStructureV3(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_chart_recognition=False,
    use_seal_recognition=False,
    text_recognition_model_name="cyrillic_PP-OCRv5_mobile_rec",
    lang="ru",
    enable_hpi=False # It slows down if enabled
)

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
/tmp/ipykernel_1230421/2637678107.py:5: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  pipeline = PPStructureV3(
/home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocBlockLayout', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory

In [3]:
from pathlib import Path
import os

PDF_GLOB="*.pdf"
Path(DATA_PATH_OCR).mkdir(exist_ok=True)

In [ ]:
from tqdm.notebook import tqdm
import shutil
from pypdfium2 import PdfiumError
documents_ocr = []
pbar = tqdm(list(Path(PROCESSED_DATA_DIR).glob(PDF_GLOB)))
for pdf in pbar:
    pbar.set_postfix_str(f"Working on {pdf}...")
    name = str(pdf).split(os.path.sep)[-1]
    data_loc = os.path.join(DATA_PATH_OCR, name.split('.')[0])
    if os.path.exists(data_loc):
        pbar.set_postfix_str(f"Skipping {pdf}...")
        continue
    data_loc_maybe = os.path.join(f"{DATA_DIR}/for_rag_2", "ocr_data", name.split('.')[0])
    if os.path.exists(data_loc_maybe):
        pbar.set_postfix_str(f"Copying existing OCR data for {pdf}...")
        shutil.copytree(data_loc_maybe, data_loc)
        continue
    #print(f"Processing {name}...")
    try:
        output = pipeline.predict(input=str(pdf))
    except Exception as e:
        print(f"Error processing {pdf}: {e}")
        if type(e) != PdfiumError:
            raise e
    #res.print()
    #print(f"Done. Saving to {data_loc}...")
    for res in output:
        res.save_to_json(save_path=data_loc)
        res.save_to_markdown(save_path=data_loc)
        res.save_to_img(save_path=data_loc)
    documents_ocr.append(output)

  0%|          | 0/2812 [00:00<?, ?it/s]

Error processing ../../data/for_rag_3/W3135393792.pdf: Failed to load document (PDFium: Data format error).


### Тестирование

In [5]:
# Определяем ground-truth

# query: [docs]

ground_truth = {
    "Give information about K theory": [
        "W1605366104.pdf",
        "QKlectures(MSJ23).pdf",
        "qkf.pdf",
        "S0894-0347-2014-00797-9.pdf",
        "a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf",
    ],
    "Write proof of the Pieri-type formula": ["W1605366104.pdf", "QKlectures(MSJ23).pdf", "S0894-0347-2014-00797-9.pdf"],
    "What does this formula mean? `v(h) < v(i) < v(l)`": ["W1605366104.pdf", "S0894-0347-2014-00797-9.pdf"],
    "Show Forbidden subsequences in chains in the k-Bruhat order": ["W1605366104.pdf", "QKlectures(MSJ23).pdf", "S0894-0347-2014-00797-9.pdf"],
    "What is `∧i(S) · det(S∨) = ∧k−i(S∨)`": ["W1605366104.pdf"],
}

#### RAGLite

In [7]:
!ollama pull qwen3:8b
!ollama pull embeddinggemma:300m

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 0800cbac9c20: 100% ▕██████████████████▏ 621 MB                         
pulling 1adbfec9dcf0: 100% ▕██████████████████▏ 8.4 KB                         
pulling 45dc10444b87: 100% ▕██████████████████▏   34 B                         
pulling 3901c6a1d7c2: 100% ▕████████████████

In [6]:
import os
from pathlib import Path
from raglite import RAGLiteConfig
from raglite import Document, insert_documents
from rerankers import Reranker

# Set Ollama API base URL
os.environ["OLLAMA_API_BASE"] = f"http://localhost:11434"  # Ensure SERVER_HOST is defined

# Configure RAGLite
# Picked from here: https://github.com/superlinear-ai/raglite/issues/85
raglite_config = RAGLiteConfig(
    db_url="duckdb:///raglite.db",
    llm=f"ollama/{MODEL_LLM}",
    embedder=f"ollama/{MODEL_EMBED}",
    reranker={
        "en": Reranker("ms-marco-MiniLM-L-12-v2", model_type="flashrank", verbose=0),  # English
        "other": Reranker("ms-marco-MultiBERT-L-12", model_type="flashrank", verbose=0),  # Other languages
    }
#    chunk_max_size=300,  # Chinese vector models are generally recommended to set around 512 context size
)

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:01<00:00, 17.6MiB/s]
ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:03<00:00, 30.2MiB/s]


In [7]:
from tqdm.notebook import tqdm

# took ~112m
raglite_documents = []
for file_name in tqdm(math_files):
    raglite_documents.append(Document.from_path(Path(os.path.join(PROCESSED_DATA_DIR, file_name))))
insert_documents(documents=raglite_documents, config=raglite_config)

  0%|          | 0/2812 [00:00<?, ?it/s]

PdfiumError: Failed to load document (PDFium: Data format error).

In [61]:
from raglite import add_context, rag, retrieve_context, vector_search

from dataclasses import replace
my_config = replace(raglite_config, search_method=vector_search)  # Or `hybrid_search`, `search_and_rerank_chunks`, ...

def rag_ask(text: str, silent: bool = False):
    response = ""

    # Retrieve relevant chunk spans with the configured search method
    chunk_spans = retrieve_context(query=text, num_chunks=5, config=my_config)

    # Append a RAG instruction based on the user prompt and context to the message history
    messages = []  # Or start with an existing message history
    messages.append(add_context(user_prompt=text, context=chunk_spans))

    # Stream the RAG response and append it to the message history
    stream = rag(messages, config=my_config)
    for update in stream:
        if not silent:
            print(update, end="")
        response += update

    # Access the documents referenced in the RAG context
    documents = [chunk_span.document for chunk_span in chunk_spans]
    docs_uniq = []
    for doc in documents:
        if doc not in docs_uniq:
            docs_uniq.append(doc)
    if not silent:
        for doc in docs_uniq:
            print(f"document: {doc}")

    return response, docs_uniq

In [62]:
test_res, test_docs = rag_ask("Give information about K theory. Find all relevant documents.")
for doc in test_docs:
    print(f"Relevant document: {doc.filename}")

梨
Okay, let's tackle this query about K theory. The user wants information on K theory and all relevant documents from the provided context. First, I need to understand what K theory is. From what I remember, K theory in mathematics is a branch that studies vector bundles on topological spaces. It has applications in algebraic geometry, topology, and even physics. But I should verify this with the given documents.

Looking through the context, there are several documents mentioning K theory. The first one is "NOTES ON QUANTUM K THEORY 51" which seems to be a section from a paper discussing quantum K theory. The user might be interested in both classical and quantum K theory. 

The documents reference authors like Anders Buch, Leonardo Mihalcea, and others. They mention topics like quantum K-theory of Grassmannians, cominuscule varieties, and Toda-type presentations. There's also a mention of equivariant cohomology and its relation to K theory. 

I need to check if there are any documen

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
raglite_mrr_all = []
raglite_ndcg_all = []
raglite_recall_all = []
raglite_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = rag_ask(req + FIND_ALL_DOCS_POSTFIX, silent=True)
    retrieved_doc_names = [doc.filename for doc in documents]
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    raglite_mrr_all.append(mrr)
    raglite_ndcg_all.append(ndcg_score)
    raglite_recall_all.append(recall)
    raglite_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall RAGLite nDCG@{COEF_K}: {np.mean(raglite_ndcg_all)}")
print(f"Overall RAGLite MRR: {np.mean(raglite_mrr_all)}")
print(f"Overall RAGLite recall@{COEF_K}: {np.mean(raglite_recall_all)}")
print(f"Overall RAGLite precision@{COEF_K}: {np.mean(raglite_precision_all)}")
# took ~1m 20s

  0%|          | 0/5 [00:00<?, ?it/s]

Request: Give information about K theory
Retrieved documents: ['W4393038435.pdf', 'QKlectures(MSJ23).pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.21398626473452756
MRR: 0.5
recall@5, precision@5: 0.2, 0.2
-----
Request: Write proof of the Pieri-type formula
Retrieved documents: ['W3118338062.pdf', 'QKlectures(MSJ23).pdf', 'W4367604437_2.pdf', 'W1985764415.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.2960819109658652
MRR: 0.5
recall@5, precision@5: 0.3333333333333333, 0.2
-----
Request: What does this formula mean? `v(h) < v(i) < v(l)`
Retrieved documents: ['W2139798975.pdf', 'W2890918596_1.pdf', 'W2122915191.pdf']
Ground truth documents: ['W1605366104.pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0

#### LightRAG

In [64]:
# From https://github.com/leovianaf/light-rag-tutorial/blob/main/notebooks/light_rag_example.ipynb
# and https://www.kaggle.com/code/parthsanghavi017/evaline-lightrag
import logging
from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from functools import partial
import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(format="%(levelname)s:%(message)s", level=logging.INFO)

rag = LightRAG(
    working_dir="./lightrag_data",
    llm_model_func=ollama_model_complete,
    llm_model_name=MODEL_LLM,
    llm_model_kwargs={"host": "http://localhost:11434", "options": {"num_ctx": 32678}, "timeout": 300},
    llm_model_max_async=1,
    embedding_func_max_async=1,
    embedding_func=EmbeddingFunc(
        embedding_dim=768,
        max_token_size=8192,
        func=partial(
            ollama_embed.func,
            embed_model=MODEL_EMBED,
#            options={"num_thread": 2},
            host="http://localhost:11434"
        )
    ),
    default_embedding_timeout=180,
)

await rag.initialize_storages()

INFO: [] Loaded graph from ./lightrag_data/graph_chunk_entity_relation.graphml with 90 nodes, 0 edges
INFO:nano-vectordb:Load (90, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_entities.json'} 90 data
INFO:nano-vectordb:Load (0, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Load (233, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_chunks.json'} 233 data


In [ ]:
# took ~30m
import os
from tqdm.notebook import tqdm
for file in tqdm(math_files):
    print(f"Adding document: {file}")
    rag.insert(os.path.join(PROCESSED_DATA_DIR, file))

  0%|          | 0/231 [00:00<?, ?it/s]

INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-6ada3202a2f354f057b77629efb1cfab
INFO: Embedding func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)


Adding document: W2957174555_1.pdf
Adding document: W2796609034.pdf
Adding document: W4313001346.pdf
Adding document: W4377086487_5.pdf
Adding document: W2465613768.pdf
Adding document: W4289128375.pdf
Adding document: W4226456969_2.pdf
Adding document: W4317037187_2.pdf
Adding document: W4283689522.pdf
Adding document: W2164376650_2.pdf


INFO:  == LLM cache == saving: default:extract:e9891b8809da2f4b10001daebf36ca26
INFO:  == LLM cache == saving: default:extract:0f4c0d8be25db7c12a0fd9f41ab85d29
INFO: Chunk 1 of 1 extracted 0 Ent + 0 Rel chunk-6ada3202a2f354f057b77629efb1cfab
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 0 entities from doc-6ada3202a2f354f057b77629efb1cfab (async: 2)
INFO: Phase 2: Processing 0 relations from doc-6ada3202a2f354f057b77629efb1cfab (async: 2)
INFO: Phase 3: Updating final 0(0+0) entities and  0 relations from doc-6ada3202a2f354f057b77629efb1cfab
INFO: Completed merging: 0 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 10 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process
INFO: No documents to process


Adding document: W4206656803.pdf
Adding document: W4324126587_2.pdf
Adding document: W1603196302_1.pdf
Adding document: W2158760784.pdf
Adding document: W2519367019.pdf
Adding document: W2108242840.pdf
Adding document: W2999061577_2.pdf
Adding document: W25036734.pdf
Adding document: W4285891493.pdf
Adding document: W4389349667.pdf


...

INFO:  == LLM cache == saving: default:extract:3443e3cbb49bc4e6db72ec3be20252b8
INFO:  == LLM cache == saving: default:extract:c0604c8f5460957ef396a93c2773d410
INFO: Chunk 1 of 1 extracted 1 Ent + 0 Rel chunk-e08cdb033e137013115ed35c10df6e7f
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 1 entities from doc-e08cdb033e137013115ed35c10df6e7f (async: 2)
INFO: Phase 2: Processing 0 relations from doc-e08cdb033e137013115ed35c10df6e7f (async: 2)
INFO: Phase 3: Updating final 1(1+0) entities and  0 relations from doc-e08cdb033e137013115ed35c10df6e7f
INFO: Completed merging: 1 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 89 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-20ee514dbb5bcbee2ca0c23cb91f2732


Adding document: W2163253099.pdf


INFO:  == LLM cache == saving: default:extract:9342a3e748126afad9073154d96e2484
INFO:  == LLM cache == saving: default:extract:717a31c96281aabbd5b9b1a2b1c87ebb
INFO: Chunk 1 of 1 extracted 1 Ent + 0 Rel chunk-20ee514dbb5bcbee2ca0c23cb91f2732
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 1 entities from doc-20ee514dbb5bcbee2ca0c23cb91f2732 (async: 2)
INFO: Phase 2: Processing 0 relations from doc-20ee514dbb5bcbee2ca0c23cb91f2732 (async: 2)
INFO: Phase 3: Updating final 1(1+0) entities and  0 relations from doc-20ee514dbb5bcbee2ca0c23cb91f2732
INFO: Completed merging: 1 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 90 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: No documents to process
INFO: No documents to process


Adding document: a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
Adding document: qkf.pdf


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-18220' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-18221' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.12/selectors.py:351: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def register(self, fileobj, events, data=None):
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-18221' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup

In [ ]:
await rag.finalize_storages()

In [ ]:
from lightrag.base import QueryParam
import re

resp = rag.query(
            "Give information about K theory. Find all relevant documents.",
            param=QueryParam(mode="hybrid", stream=False),
        )
display(resp)
[ x.split("] ", 1)[1] for x in re.findall(r'- \[\d\] .+\.pdf', str(resp)) ]

INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: hybrid:keywords:10768ae6d41b67b561827bc46f1848f5
INFO: Embedding func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO: Query edges: K theory (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


"Sorry, I'm not able to provide an answer to that question.[no-context]"

[]

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-39236' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-39237' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.12/asyncio/events.py:36: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def __init__(self, callback, args, loop, context=None):
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-39237' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[T

In [79]:
def lightrag_ask(prompt: str, silent=False, mode="hybrid"):
    resp = rag.query(prompt, param=QueryParam(mode=mode, stream=False))
    if not silent:
        print(resp)
    docs = [ x.split("] ", 1)[1] for x in re.findall(r'- \[\d+\] .+\.pdf', str(resp)) ]
    uniq_docs = []
    for doc in docs:
        if doc not in uniq_docs:
            uniq_docs.append(doc)
    return resp, [ x.replace(f"{PROCESSED_DATA_DIR}/", "") for x in uniq_docs ]

In [82]:
lightrag_ask("Give information about K theory. Find all relevant documents.", silent=True)

INFO: Query nodes: Document retrieval (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Query edges: K theory, Information retrieval (top_k:40, cosine:0.2)
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == Query cache hit, using cached response as query result


('The provided context does not contain any information or references related to **K theory** (a branch of mathematics involving vector bundles, algebraic topology, or homological algebra). The **Knowledge Graph Data** and **Document Chunks** listed in the context only reference file paths for PDF documents, primarily within the `for_rag_2` directory, but none of these entries explicitly mention K theory, its applications, or related concepts. \n\n### Relevant Documents (if any)\nNo documents in the provided context are directly or indirectly linked to K theory. The listed files appear to be data artifacts or research papers in unspecified fields, but their contents are not described in the context. \n\n---\n\n### References\n- [1] S0894-0347-2014-00797-9.pdf (Research paper, but no explicit connection to K theory)  \n- [2] W2111717017_1.pdf  \n- [3] W3201116201.pdf  \n- [4] W2147684619.pdf  \n- [5] W3207743871.pdf  \n\n**Note**: The above references are derived from the Document Chunk

In [81]:
from tqdm.notebook import tqdm
import numpy as np

rag.clear_cache()

lightrag_mrr_all = []
lightrag_ndcg_all = []
lightrag_recall_all = []
lightrag_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = lightrag_ask(req + "." + FIND_ALL_DOCS_POSTFIX, silent=True)
    retrieved_doc_names = documents
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    lightrag_mrr_all.append(mrr)
    lightrag_ndcg_all.append(ndcg_score)
    lightrag_recall_all.append(recall)
    lightrag_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall LightRAG nDCG@{COEF_K}: {np.mean(lightrag_ndcg_all)}")
print(f"Overall LightRAG MRR: {np.mean(lightrag_mrr_all)}")
print(f"Overall LightRAG recall@{COEF_K}: {np.mean(lightrag_recall_all)}")
print(f"Overall LightRAG precision@{COEF_K}: {np.mean(lightrag_precision_all)}")
# took ~1 24s + cached (38)

INFO: [] Process 216158 drop llm_response_cache
INFO: Cleared all cache


  0%|          | 0/5 [00:00<?, ?it/s]

Request: Give information about K theory


INFO:  == LLM cache == saving: hybrid:keywords:10768ae6d41b67b561827bc46f1848f5
INFO: Query nodes: Document retrieval (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Query edges: K theory, Information retrieval (top_k:40, cosine:0.2)
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: hybrid:query:659fc891367c93e0134553c831fdda47


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'W2111717017_1.pdf', 'W3201116201.pdf', 'W2147684619.pdf', 'W3207743871.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.3391602052736161
MRR: 1.0
recall@5, precision@5: 0.2, 0.2
-----
Request: Write proof of the Pieri-type formula


...

Retrieved documents: []
Ground truth documents: ['W1605366104.pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: Show Forbidden subsequences in chains in the k-Bruhat order


INFO:  == LLM cache == saving: hybrid:keywords:e257923c6a67c95875e4dc92b36ee482
INFO: Query nodes: subsequences, chains, k-Bruhat order (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Query edges: Forbidden subsequences, Chains in k-Bruhat order, k-Bruhat order (top_k:40, cosine:0.2)
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: hybrid:query:3d6f079efd4be29c9b1a5a5201cde7ae


Retrieved documents: []
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: What is `∧i(S) · det(S∨) = ∧k−i(S∨)`


INFO:  == LLM cache == saving: hybrid:keywords:ad736025872c76dfab0b34437ed8d8fb
INFO: Query nodes: ∧i(S), det(S∨), ∧k−i(S∨) (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Query edges: Mathematical equation, Logical operations, Determinant calculation (top_k:40, cosine:0.2)
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: hybrid:query:b6afcfd5df2a7a89d2afb324a0d0fb89


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'QKlectures(MSJ23).pdf', 'W2480618327_2.pdf', 'W4287591918.pdf', 'W3207743871.pdf']
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Overall LightRAG nDCG@5: 0.16168778625927452
Overall LightRAG MRR: 0.4
Overall LightRAG recall@5: 0.10666666666666666
Overall LightRAG precision@5: 0.08


In [83]:
from tqdm.notebook import tqdm
import numpy as np

rag.clear_cache()

lightrag_local_mrr_all = []
lightrag_local_ndcg_all = []
lightrag_local_recall_all = []
lightrag_local_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = lightrag_ask(req + "." + FIND_ALL_DOCS_POSTFIX, silent=True, mode="local")
    retrieved_doc_names = documents
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    lightrag_local_mrr_all.append(mrr)
    lightrag_local_ndcg_all.append(ndcg_score)
    lightrag_local_recall_all.append(recall)
    lightrag_local_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall LightRAG nDCG@{COEF_K}: {np.mean(lightrag_local_ndcg_all)}")
print(f"Overall LightRAG MRR: {np.mean(lightrag_local_mrr_all)}")
print(f"Overall LightRAG recall@{COEF_K}: {np.mean(lightrag_local_recall_all)}")
print(f"Overall LightRAG precision@{COEF_K}: {np.mean(lightrag_local_precision_all)}")
# took ~1 24s + cached (38)

INFO: [] Process 216158 drop llm_response_cache
INFO: Cleared all cache


  0%|          | 0/5 [00:00<?, ?it/s]

Request: Give information about K theory


INFO:  == LLM cache == saving: local:keywords:b154f7645cf728d81a628020fb034cee
INFO: Query nodes: K theory (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: local:query:d5f028206230789ee64e449ef763cea2


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'W2147684619.pdf', 'W4360988008_3.pdf', 'QKlectures(MSJ23).pdf', 'W3201116201.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.48522855511632257
MRR: 1.0
recall@5, precision@5: 0.4, 0.4
-----
Request: Write proof of the Pieri-type formula


...

Retrieved documents: ['`W2110877747.pdf', '`W3207743871.pdf', '`W4377086487_5.pdf', '`W2111717017_1.pdf', '`W3014677203.pdf', '`W3174484797.pdf', '`W3165301745.pdf', '`W2126017743.pdf', '`W3008950759.pdf', '`W2171382235.pdf', '`W3201116201.pdf', '`W2098771155_2.pdf', '`W2168858925.pdf', '`W2051388013.pdf', '`W3201336880.pdf', '`W2988279279.pdf', '`W2017723339.pdf', '`W3164429027.pdf', '`W2827850560.pdf', '`W2560356400_3.pdf']
Ground truth documents: ['W1605366104.pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: Show Forbidden subsequences in chains in the k-Bruhat order


INFO:  == LLM cache == saving: local:keywords:203b7c295b388bc4a4f9d016b7731317
INFO: Query nodes: chains, k-Bruhat order (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: local:query:09f982542f0b2a3b97b8ffd786a17df0


Retrieved documents: ['QKlectures(MSJ23).pdf', 'W2147684619.pdf', 'W2111717017_1.pdf', 'W2110877747.pdf', 'S0894-0347-2014-00797-9.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.6508205185601091
MRR: 1.0
recall@5, precision@5: 0.6666666666666666, 0.4
-----
Request: What is `∧i(S) · det(S∨) = ∧k−i(S∨)`


INFO:  == LLM cache == saving: local:keywords:e887a2f2df174855df9dc0aeede93f2e
INFO: Query nodes: ∧i(S), det(S∨), ∧k−i(S∨), S, ∧, ∨, det (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 0 relations
INFO: Raw search results: 40 entities, 0 relations, 0 vector chunks
INFO: After truncation: 40 entities, 0 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 40 entities, 0 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8 E1/9 E1/10 E1/11 E1/12 E1/13 E1/14 E1/15 E1/16 E1/17 E1/18 E1/19 E1/20
INFO:  == LLM cache == saving: local:query:9ba06e2c6ef432f2476b8fc73335796d


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'QKlectures(MSJ23).pdf', 'W4287591918.pdf', 'W2110877747.pdf', 'W4317037187_2.pdf']
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Overall LightRAG nDCG@5: 0.22720981473528634
Overall LightRAG MRR: 0.4
Overall LightRAG recall@5: 0.21333333333333332
Overall LightRAG precision@5: 0.16


In [84]:
from tqdm.notebook import tqdm
import numpy as np

rag.clear_cache()

lightrag_global_mrr_all = []
lightrag_global_ndcg_all = []
lightrag_global_recall_all = []
lightrag_global_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = lightrag_ask(req + "." + FIND_ALL_DOCS_POSTFIX, silent=True, mode="global")
    retrieved_doc_names = documents
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    lightrag_global_mrr_all.append(mrr)
    lightrag_global_ndcg_all.append(ndcg_score)
    lightrag_global_recall_all.append(recall)
    lightrag_global_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall LightRAG nDCG@{COEF_K}: {np.mean(lightrag_global_ndcg_all)}")
print(f"Overall LightRAG MRR: {np.mean(lightrag_global_mrr_all)}")
print(f"Overall LightRAG recall@{COEF_K}: {np.mean(lightrag_global_recall_all)}")
print(f"Overall LightRAG precision@{COEF_K}: {np.mean(lightrag_global_precision_all)}")
# took ~1 24s + cached (38)

INFO: [] Process 216158 drop llm_response_cache
INFO: Cleared all cache


  0%|          | 0/5 [00:00<?, ?it/s]

Request: Give information about K theory


INFO:  == LLM cache == saving: global:keywords:a98a065f691e60729409f4ec34014282
INFO: Query edges: K theory, Information retrieval (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


Retrieved documents: []
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: Write proof of the Pieri-type formula


...

Retrieved documents: []
Ground truth documents: ['W1605366104.pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: Show Forbidden subsequences in chains in the k-Bruhat order


INFO:  == LLM cache == saving: global:keywords:374dced9d594f6a043d68d9a4608370e
INFO: Query edges: Forbidden subsequences, Chains, k-Bruhat order (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


Retrieved documents: []
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'S0894-0347-2014-00797-9.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: What is `∧i(S) · det(S∨) = ∧k−i(S∨)`


INFO:  == LLM cache == saving: global:keywords:75c924ce9c53199291982d6b14b08a6e
INFO: Query edges: Mathematical equation, Symbolic logic, Determinant notation (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.


Retrieved documents: []
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Overall LightRAG nDCG@5: 0.0
Overall LightRAG MRR: 0.0
Overall LightRAG recall@5: 0.0
Overall LightRAG precision@5: 0.0
